In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.parser import parse_bgl_fields, build_miner, mine_templates
from src.features import build_windows, attach_window_labels, count_vector, apply_tfidf

df = parse_bgl_fields('../data/raw/BGL_full/BGL.log')

Parsed: 4747963 | Skipped: 0


In [2]:
miner = build_miner('../drain3_bgl.ini', '../models/bgl_drain_state_full.bin')
df = mine_templates(df, miner)

print("Templates:", df['EventId'].nunique())

Templates: 1124


In [3]:
windows = build_windows(df, window_size=100)
windows = attach_window_labels(windows, df, window_size=100)

print("Windows:", len(windows))
print("Anomalous windows:", int(windows['y'].sum()), "/", len(windows),
      f"({windows['y'].mean() * 100:.2f}%)")

Windows: 47480
Anomalous windows: 4803 / 47480 (10.12%)


In [4]:
X_counts, event_names = count_vector(windows)
X = apply_tfidf(X_counts)
y = windows['y'].values

np.save('../data/features/X_counts_full_bgl.npy', X_counts)
np.save('../data/features/X_full_bgl.npy', X)
np.save('../data/features/y_full_bgl.npy', y)
windows[['WindowId', 'y', 'causes']].to_csv('../data/features/windows_bgl.csv', index=False)

print("Feature matrix:", X_counts.shape, "| Anomalous:", int(y.sum()))

Feature matrix: (47480, 1124) | Anomalous: 4803


In [ ]:
template_map = {c.cluster_id: c.get_template() for c in miner.drain.clusters}
assert len(event_names) == X_counts.shape[1], "Mismatch between templates and matrix"

template_texts = [template_map[e] or '(empty message)' for e in event_names]

pd.DataFrame({'EventId': event_names, 'EventTemplate': template_texts}
             ).to_csv('../data/parsed/templates_full_bgl.csv', index=False)

print("Saved", len(event_names), "templates")

Saved 1124 templates


In [ ]:

labeled_lines = df[df['label'] != '-'][['EventId', 'label', 'content']].copy()
labeled_lines['EventTemplate'] = labeled_lines['EventId'].map(template_map)
labeled_lines.to_csv('../data/parsed/bgl_labeled_lines.csv', index=False)

print("Labeled lines:", len(labeled_lines), "| Distinct fault categories:",
      labeled_lines['label'].nunique())
print(labeled_lines['label'].value_counts())

Labeled lines: 348460 | Distinct fault categories: 41
label
KERNDTLB     152734
KERNSTOR      63491
APPSEV        49651
KERNMNTF      31531
KERNTERM      23338
KERNREC        6145
APPREAD        5983
KERNRTSP       3983
APPRES         2370
APPUNAV        2048
APPTO          1991
KERNMICRO      1503
APPOUT          816
KERNMNT         720
APPBUSY         512
KERNMC          342
APPCHILD        320
KERNSOCK        209
KERNPOW         192
LINKIAP         166
APPALLOC        144
KERNSERV         94
MASABNORM        37
LINKDISC         24
KERNPAN          18
KERNCON          16
KERNNOETH        14
LINKPAP          14
MONPOW           12
MASNORM          10
APPTORUS         10
KERNPROG          5
KERNFLOAT         3
KERNRTSA          3
MMCS              3
MONNULL           2
LINKBLL           2
KERNEXT           1
KERNBIT           1
MONILL            1
KERNTLBE          1
Name: count, dtype: int64
